# 第 25 章 · Web：从检索结果到可核对引用

检索命中、打开的正文、答案里的引文和引文是否支持结论，是四个问题。引用首先要能回到本次实际取得的内容版本；匹配一段原文仍不等于证明整句推断。

本章运行本地响应解析、引用定位与取消协议。`teaching_text_v1` 是教学适配格式，不是 Codex 或搜索服务的线协议；没有发网络请求。

**运行范围：局部机制实验。** 按顺序执行，每个实验都通过真实函数、文件或队列产生结果，再用断言检查。修改参数后重跑该实验，必要时先重启内核并执行全部。所有合成输入都只证明本地机制；本章没有累计检查点，真实服务与设备仍待验证。

本章使用原作者的设计与局部原型；下面明确写出的简化是教材选择，不是 Codex 原样实现。固定源码：`c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee`。

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter

## 1 · snippet 与 page_window 保留不同证据资格

同一个 ref_id 可以出现在不同请求；完整 key 必须包含 session/request。下方先给摘要，再给正文窗口。摘要能定位原文，但不能满足 require_page=True。

In [2]:
import json
m = load_example('25', 'citations')
table = m.CitationTable('reader-session')
acquired_at = '2026-01-01T00:00:00+00:00'  # 合成响应的固定实验时间，不是实时抓取时间。
def response(kind='page_window', text='筛选使用 status 字段。\n缺失字段返回空列表。', line_start=10):
    return json.dumps({'output': '受控搜索观察', 'results': [{
        'type': 'teaching_text_v1', 'ref_id': 'ref1', 'url': 'https://example.invalid/task-list',
        'title': '合成任务文档', 'kind': kind, 'line_start': line_start, 'text': text}]}, ensure_ascii=False).encode()
table.begin('search-1')
snippet_obs = table.ingest('search-1', response('snippet'), acquired_at)
snippet = table.find(snippet_obs.sources[0], 'status')[0]
try:
    table.resolve_citation(snippet)
except m.EvidenceError as error:
    assert str(error) == 'page_not_opened'
else:
    raise AssertionError('摘要不能冒充已打开正文')
table.begin('open-1')
page_obs = table.ingest('open-1', response(), acquired_at)
assert page_obs.sources[0] != snippet_obs.sources[0]
print({'snippet_key': snippet_obs.sources[0], 'page_key': page_obs.sources[0]})

{'snippet_key': SourceKey(session_id='reader-session', request_id='search-1', ref_id='ref1'), 'page_key': SourceKey(session_id='reader-session', request_id='open-1', ref_id='ref1')}


## 2 · 引文绑定内容摘要和精确区间

find 产出的字符范围采用 Python Unicode code point，并计算绝对行号。改变 pattern，观察 quote 和行范围同步变化。伪造引文或借用另一个内容摘要都要拒绝。

In [3]:
from dataclasses import replace
citations = table.find(page_obs.sources[0], '缺失字段返回空列表。')
assert len(citations) == 1
citation = citations[0]
resolved = table.resolve_citation(citation)
assert resolved['line_start'] == 11
assert resolved['support_status'] == 'exact_span_only'
for changed, reason in [(replace(citation, quote='不存在的原文'), 'quote_mismatch'),
                        (replace(citation, content_digest='old-digest'), 'content_version_mismatch')]:
    try:
        table.resolve_citation(changed)
    except m.EvidenceError as error:
        assert str(error) == reason
    else:
        raise AssertionError(reason)
print(resolved)

{'source': {'session_id': 'reader-session', 'request_id': 'open-1', 'ref_id': 'ref1'}, 'url': 'https://example.invalid/task-list', 'title': '合成任务文档', 'acquired_at': '2026-01-01T00:00:00+00:00', 'content_kind': 'page_window', 'content_digest': 'bfb355ebae6ac29df22294a490a84c6bcb859360cc3c3c283c3fea3379e6aad7', 'line_start': 11, 'line_end': 11, 'quote': '缺失字段返回空列表。', 'support_status': 'exact_span_only', 'trust': 'external_data'}


## 3 · 空结果、缺结果、未知格式和传输失败各自保留

未知 provider 结构必须作为 opaque data 保留，不能把任意字符串当成可信引用。下面用一组参数逐个调用真实解析器，而非直接生成期待状态。

In [4]:
states = {}
for key, raw in [('absent', {'output': '没有 result 字段'}),
                 ('empty', {'output': '空列表', 'results': []}),
                 ('opaque', {'output': '未知格式', 'results': [{'type': 'future_v9', 'body': '任意外部内容'}]})]:
    table.begin(key)
    observation = table.ingest(key, json.dumps(raw).encode(), acquired_at)
    states[key] = observation.result_state
    if key == 'opaque':
        assert not observation.sources and len(observation.opaque_results) == 1
assert states == {'absent': 'absent', 'empty': 'empty', 'opaque': 'present'}
table.begin('failed-transport')
failed = table.fail('failed-transport', 'timeout')
assert failed.status == 'failed' and failed.error_code == 'timeout'
print({'result_states': states, 'transport': failed.status})

{'result_states': {'absent': 'absent', 'empty': 'empty', 'opaque': 'present'}, 'transport': 'failed'}


## 4 · 取消关闭入口，历史引用仍可核对

晚到响应不能进入当前 evidence table。关闭会话也不删除已经取得的历史证据；恢复时应核对旧版本，不能把旧 acquired_at 改成今天假装刷新。

In [5]:
table.begin('cancelled-request')
table.cancel('cancelled-request')
try:
    table.ingest('cancelled-request', response(), acquired_at)
except m.EvidenceError as error:
    assert str(error) == 'late_or_unknown_request'
else:
    raise AssertionError('取消后的回包不应发布')
table.close()
assert table.resolve_citation(citation)['acquired_at'] == acquired_at
try:
    table.begin('new-request')
except m.EvidenceError as error:
    assert str(error) == 'session_closed'
else:
    raise AssertionError('关闭后不应接纳新请求')
print({'historical_quote': table.resolve_citation(citation)['quote'], 'refresh': 'not_performed'})

{'historical_quote': '缺失字段返回空列表。', 'refresh': 'not_performed'}


## 源码对照与接回主线

以下为固定提交的已定位源码入口。阅读源码能解释设计边界，不等于执行过其测试；本章 Python 原型与这些 Rust 接口不是同一实现。

- [codex-rs/core/src/tools/spec_plan.rs#L724-L754](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/tools/spec_plan.rs#L724-L754)
- [codex-rs/protocol/src/models.rs#L1341-L1366](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/protocol/src/models.rs#L1341-L1366)
- [codex-rs/ext/web-search/src/tool.rs#L108-L177](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/ext/web-search/src/tool.rs#L108-L177)

**接回条件：** 将局部机制接入同一 Dispatcher、Scheduler、History 和受保护验证器，再执行原合同。目录、断言和演示输出都不能替代累计集成证据。